In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.timesfm.circuitlens import CircuitLensTimesFM
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    load_dyst_samples,
)


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
apply_custom_style("../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

### Load Data

In [ ]:
split_name = "gift-eval"

subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Lorenz"
    # ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    # Prepare input series
    sample_idx = 0
    selected_dim = 0
    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]

    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    # Load the path to gift-eval data, which is stored in .env file
    split_dir = os.path.join(WORK_DIR, DATA_DIR, split_name)
    # system_name = "electricity/D"
    system_name = "m4_monthly"

    to_univariate = False
    term = "long"

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    print("Dataset frequency: ", dataset.freq)
    print("Prediction length: ", dataset.prediction_length)
    print("Number of windows in the rolling evaluation: ", dataset.windows)

    test_split_iter = dataset.test_data
    test_data = next(iter(test_split_iter))

    test_input_split_iter = dataset.test_data.input

    input = next(iter(test_input_split_iter))
    print("Keys in the test data: ", input.keys())

    print("\n\nContext Item id: ", input["item_id"])
    print("Context Start Date: ", input["start"])
    print("Context Frequency: ", input["freq"])
    print(f"Context shape: {input['target'].shape}")

    test_label_split_iter = dataset.test_data.label
    label = next(iter(test_label_split_iter))
    print("\n\nForecast Item id: ", label["item_id"])
    print("Forecast Start Date: ", label["start"])
    print("Forecast Frequency: ", label["freq"])
    print(f"Forecast shape: {label['target'].shape}")

    fig, ax = plt.subplots(figsize=(5, 2))
    context = to_pandas(test_data[0])
    future_vals = to_pandas(test_data[1])
    prediction_length = future_vals.shape[0]
    print(f"prediction length: {prediction_length}")
    context.plot(color="black", linewidth=1, ax=ax)
    future_vals.plot(color="tab:blue", linewidth=1, ax=ax)
    ax.grid(which="both")
    # ax.legend("ground truth", loc="upper left")
    plt.show()
    num_nans_context = context.isna().sum()
    num_nans_future_vals = future_vals.isna().sum()

    context = torch.tensor(context).unsqueeze(0)
    future_vals = torch.tensor(future_vals).unsqueeze(0)
    print(f"num nans context: {num_nans_context}")
    print(f"num nans future vals: {num_nans_future_vals}")
    print(f"context length: {context.shape}")
    print(f"future vals shape: {future_vals.shape}")


### Load Pipeline

In [ ]:
pipeline = CircuitLensTimesFM("google/timesfm-2.5-200m-pytorch", device_map=device)
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

### Predictions with Attributions

In [ ]:
num_samples = 10
do_sample = True if num_samples > 1 else False

# NOTE: fix this
pipeline.remove_all_hooks()

print("Adding head attribution hooks")
pipeline.add_head_attribution_hooks(
    [(i, j) for i in range(num_layers) for j in range(num_heads)],
)

In [ ]:
pred = pipeline.predict(context=context, prediction_length=prediction_length)

### Visualize

In [ ]:
# Prepare context and predictions
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]

fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(future_timesteps, future_vals_np, color="black", linewidth=1, linestyle="--", label="Ground Truth")
for i in range(len(preds)):
    ax.plot(future_timesteps, preds[i], color=DEFAULT_COLORS[i], linewidth=1, alpha=0.5)
ax.plot(future_timesteps, np.median(preds, axis=0), color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")

# Save plot
save_fname = (
    f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name}"
)
fig.tight_layout()

In [ ]:
layer_idx = 0
head_idx = 0
rollout_idx = 0  # number of predictions of length 64

pipeline.head_attribution_outputs[layer_idx][head_idx][rollout_idx].shape

In [ ]:
head_outputs = {
    i: [collect_attributions(pipeline.head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

In [ ]:
len(head_outputs[0])

In [ ]:
head_outputs[0][0].shape

In [ ]:
layer_idx = 0
head_idx = 0
sample_idx = 0
timestep_idx = 0

In [ ]:
# Normalize head outputs
def normalize_outputs(outputs):
    return {layer: [h / torch.norm(h, dim=-1, keepdim=True) for h in outputs[layer]] for layer in outputs}


head_outputs_normalized = normalize_outputs(head_outputs)
num_samples, num_timesteps, d_model = head_outputs[0][0].shape

# Calculate gramians
sa_gramians = torch.zeros(num_layers, num_samples, num_timesteps, num_heads, num_heads)

for layer in range(num_layers):
    for h1, h2 in [(i, j) for i in range(num_heads) for j in range(num_heads)]:
        sa_gramians[layer, :, :, h1, h2] = torch.einsum(
            "b t d, b t d -> b t", head_outputs_normalized[layer][h1], head_outputs_normalized[layer][h2]
        )

In [ ]:
# Compute SVD of gramians and analyze head interactions
sa_singular_values = torch.zeros(num_layers, num_samples, num_timesteps, num_heads)

for layer in range(num_layers):
    for sample in range(num_samples):
        for timestep in range(num_timesteps):
            sa_singular_values[layer, sample, timestep] = torch.linalg.svd(sa_gramians[layer, sample, timestep]).S

# Calculate entropic rank
sa_p_vals = torch.sqrt(sa_singular_values) / (torch.sqrt(sa_singular_values).sum(dim=-1, keepdim=True))
sa_entropy = -torch.sum(sa_p_vals * torch.log(sa_p_vals), dim=-1)
sa_mean_entropy = torch.mean(sa_entropy, dim=(1, 2))

# Plot entropic ranks
plt.figure(figsize=(5, 5))
plt.plot(torch.exp(sa_mean_entropy), label="SA Heads", linewidth=2)
plt.xlabel("Layer", fontweight="bold")
plt.xticks(range(num_layers))
plt.ylabel("Entropic Rank $\\exp\\left(-\\sum_{i=1}^{n} p_i \\log(p_i)\\right)$", fontweight="bold")
plt.title("Entropic Rank of Head Outputs", fontweight="bold")
plt.legend()
# plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def plot_head_similarity_with_magnitude(head_outputs, output_type, save_path=None):
    """Plot head similarity and magnitude heatmaps for each layer."""
    n_layers = 20
    n_heads = 16
    fig, axs = plt.subplots(4, 5, figsize=(20, 16))
    axs = axs.flatten()

    for layer_idx in range(n_layers):
        heads_data = head_outputs[layer_idx]
        similarities = np.zeros((n_heads, n_heads))
        magnitudes = np.zeros(n_heads)

        # Calculate magnitudes
        for i in range(n_heads):
            head_output = heads_data[i][0]
            magnitudes[i] = torch.norm(head_output, dim=-1).mean().item()

        # Calculate similarities
        for i in range(n_heads):
            for j in range(n_heads):
                i_flat = heads_data[i][0].reshape(-1, heads_data[i][0].shape[-1])
                j_flat = heads_data[j][0].reshape(-1, heads_data[j][0].shape[-1])
                similarities[i, j] = torch.nn.functional.cosine_similarity(i_flat, j_flat, dim=-1).mean().item()

        # Plot
        ax = axs[layer_idx]
        im = ax.imshow(similarities, cmap="RdBu", vmin=-1, vmax=1)
        ax.set_title(f"Layer {layer_idx}")
        ax.set_xlabel("Head Index")
        ax.set_ylabel("Head Index")

        # Add magnitude bars
        divider = make_axes_locatable(ax)
        ax_mag = divider.append_axes("right", size="15%", pad=0.1)
        ax_mag.barh(np.arange(n_heads), magnitudes, color="coral")
        ax_mag.set_ylim(ax_mag.get_ylim()[::-1])
        ax_mag.set_title("Magnitude")
        ax_mag.set_yticks([])

    plt.tight_layout()
    fig.colorbar(im, ax=axs, label="Cosine Similarity", shrink=0.6)
    fig.suptitle(f"{output_type.upper()} Head Output Similarity and Magnitude", fontsize=16)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


# Plot both CA and SA head outputs
plot_save_dir = "../figs/head_outputs"
fig_sa = plot_head_similarity_with_magnitude(head_outputs, "sa", f"{plot_save_dir}/sa_head_similarity.png")
plt.show()